# BESS optimal placement and sizing

Run this after candidate site selection and prepare a completed hourly system profile.

In [1]:
import json
import sys
from pathlib import Path
import pandas as pd
from IPython.display import display

ROOT = next(
    (parent for parent in (Path.cwd(), *Path.cwd().parents)
     if (parent / "src" / "paths.py").is_file()),
    None,
)
if ROOT is None:
    raise FileNotFoundError("Could not locate project root containing src/paths.py")
sys.path.insert(0, str(ROOT))
from src.paths import CONFIG, OPTIMIZATION, PROCESSED
from src.bess_optimization import optimize_bess

## Load inputs and solve

The optimizer requires ranked eligible sites and an hourly profile with `timestamp`, `demand_mw`, and `solar_mw`.

In [3]:
profile_options = [
    PROCESSED / "optimization_profile.csv",
    PROCESSED / "system_hourly_profile.csv",
]
profile_path = next((path for path in profile_options if path.is_file()), None)
candidates_path = PROCESSED / "candidate_bess_sites.csv"

profile = pd.read_csv(profile_path) if profile_path else pd.DataFrame()
candidates = pd.read_csv(candidates_path) if candidates_path.is_file() else pd.DataFrame()

if not profile.empty and "timestamp" in profile.columns:
    profile["timestamp"] = pd.to_datetime(profile["timestamp"], errors="coerce", utc=True)

config = json.loads((CONFIG / "model_config.json").read_text(encoding="utf-8"))

has_valid_profile = (
    not profile.empty
    and {"demand_mw", "solar_mw"}.issubset(profile.columns)
)

has_eligible_candidates = (
    not candidates.empty
    and ("eligible" not in candidates.columns or candidates["eligible"].astype(bool).any())
)

inputs_ready = has_valid_profile and has_eligible_candidates

if not inputs_ready:
    print("Optimization inputs are not ready:")
    if profile_path is None or profile.empty:
        print(f"- Missing hourly profile: provide {PROCESSED / 'optimization_profile.csv'} (or system_hourly_profile.csv), based on datasets/templates/system_hourly_profile_template.csv.")
    elif not {"demand_mw", "solar_mw"}.issubset(profile.columns):
        print("- Hourly profile is missing required columns: demand_mw and solar_mw.")
    if not candidates_path.is_file() or candidates.empty:
        print(f"- Candidate site file is missing or empty: {candidates_path}. Run notebook/06_BESS_Candidate_Site_Selection.ipynb after loading grid data.")
    elif not has_eligible_candidates:
        print("- Candidate site file contains no eligible substations (eligible == True). Check voltage screening in notebook 06.")
else:
    print("Profile records:", len(profile))
    eligible_count = len(candidates[candidates["eligible"].astype(bool)]) if "eligible" in candidates.columns else len(candidates)
    print(f"Candidate substations: {len(candidates)} total ({eligible_count} eligible for BESS)")
results = None


Profile records: 744
Candidate substations: 30 total (30 eligible for BESS)


### Run optimization

In [5]:
if inputs_ready:
    results = optimize_bess(profile=profile, candidates=candidates, config=config)
    print("Optimization completed.")
    print("Objective value:", results["objective_value"])
    print("Peak residual demand:", results["peak_residual_mw"], "MW")
else:
    print("Optimization skipped until the required inputs are populated.")

Running HiGHS 1.15.1 (git hash: 04024d7): Copyright (c) 2026 under MIT licence terms
Includes third-party software components, see THIRD_PARTY_NOTICES.md for full details
MIP has 91604 rows; 68539 cols; 271056 nonzeros; 30 integer variables (30 binary)
Coefficient ranges:
  Matrix  [9e-01, 1e+02]
  Cost    [4e+01, 5e+04]
  Bound   [1e+00, 1e+00]
  RHS     [5e+00, 3e+03]
Presolving model
68510 rows, 45445 cols, 180972 nonzeros 0s
56104 rows, 33039 cols, 170691 nonzeros 0s
48696 rows, 25631 cols, 202363 nonzeros 0s
48686 rows, 25621 cols, 202437 nonzeros 1s
Presolve reductions: rows 48686(-42918); columns 25621(-42918); nonzeros 202437(-68619) 

Solving MIP model with:
   48686 rows
   25621 cols (30 binary, 0 integer, 0 implied int., 25591 continuous, 0 domain fixed)
   202437 nonzeros
   Thread count 10 (of 20 threads). Using 1 max workers. Parallel search off

Src: B => Branching; C => Central rounding; F => Feasibility pump; H => Heuristic;
     I => Shifting; J => Feasibility jump; 

### Display selected locations

In [7]:
if results is not None:
    selected_sites = results["selected_sites"]
    display(selected_sites)
else:
    selected_sites = pd.DataFrame(columns=["bus_id", "power_mw", "energy_mwh"])
    print("No selected sites to display because optimization did not run.")

Empty DataFrame
Columns: [bus_id, power_mw, energy_mwh, site_cost_factor]
Index: []


### Display battery sizes

In [9]:
if results is None:
    print("Battery sizing is unavailable until optimization runs.")
elif not selected_sites.empty:

    total_power = selected_sites["power_mw"].sum()

    total_energy = selected_sites["energy_mwh"].sum()

    print("Total Battery Power:", total_power, "MW")
    print("Total Battery Energy:", total_energy, "MWh")

else:

    print("The current economic assumptions did not select any battery sites.")

The current economic assumptions did not select any battery sites.


### Save optimization results

In [11]:
if results is None:
    print("No optimization results to save.")
else:
    OPTIMIZATION.mkdir(parents=True, exist_ok=True)
    selected_sites.to_csv(OPTIMIZATION / "optimal_bess_locations.csv", index=False)
    results["dispatch"].to_csv(OPTIMIZATION / "battery_dispatch_results.csv", index=False)
    summary = pd.DataFrame([{
        "objective_value": results["objective_value"],
        "peak_residual_mw": results["peak_residual_mw"],
        "number_of_bess_sites": len(selected_sites),
        "total_power_mw": selected_sites["power_mw"].sum() if not selected_sites.empty else 0,
        "total_energy_mwh": selected_sites["energy_mwh"].sum() if not selected_sites.empty else 0
    }])
    summary.to_csv(OPTIMIZATION / "optimization_summary.csv", index=False)
    display(summary)

   objective_value  peak_residual_mw  ...  total_power_mw  total_energy_mwh
0              0.0            2650.0  ...               0                 0

[1 rows x 5 columns]
